In [ ]:
import os
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import seaborn as sns

from matplotlib import colors, pyplot as plt, ticker
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import cmcrameri.cm as cmc

%matplotlib inline

## Scaling

### IsoFLOP profiles

For every fixed compute budget, fit

$$
\log_{10} L = a\left(\log_{10} N\right)^2 + b\log_{10}N + c,
$$

where $N$ is model size and $L$ is either validation or training loss. The vertex gives the compute-optimal model size. Its paired training-token count is obtained by log-log interpolation within the same fixed-compute slice. If a vertex falls outside the tested model-size range, the reported optimum is clipped to the nearest tested boundary and flagged in the summary.

The custom ViT uses $256\times256$ images and $16\times16$ patches with no class token, giving $(256/16)^2=256$ tokens per image. For the IsoFLOP runs, each iteration contains 512 images, so one iteration is 131,072 tokens. One pass through the training data (85,693 iterations) is 11,231,952,896 tokens.

This follows the IsoFLOP-profile method in Figure 3 of [Hoffmann et al. (2022)](https://arxiv.org/abs/2203.15556). The palette uses Seaborn's reversed `mako` map to reproduce Figure 3's light-green-to-dark-purple progression without a low-contrast yellow curve.

In [ ]:
# Resolve paths whether the notebook is launched from the repository root or scripts/.
ROOT = Path.cwd()
if not (ROOT / "workspace" / "paper").exists():
    ROOT = ROOT.parent

PAPER_DIR = ROOT / "workspace" / "paper"
POINTS_CSV = PAPER_DIR / "isoflop_points.csv"
METRICS = {
    "validation": ("val_loss", "Validation loss"),
    "training": ("train_loss", "Training loss"),
}
REQUIRED_COLUMNS = {
    "target_flops",
    "params_m",
    "iter",
    "train_loss",
    "val_loss",
}

IMAGE_SIZE = 256
PATCH_SIZE = 16
TOKENS_PER_IMAGE = (IMAGE_SIZE // PATCH_SIZE) ** 2
IMAGES_PER_ITERATION = 512
TOKENS_PER_ITERATION = TOKENS_PER_IMAGE * IMAGES_PER_ITERATION
ONE_EPOCH_ITERATIONS = 85_693
ONE_EPOCH_TOKENS = ONE_EPOCH_ITERATIONS * TOKENS_PER_ITERATION
FIGURE_SIZE = (6.5, 2.5)
FIGURE_SCALE = 0.5
MARKER_AREA_SCALE = FIGURE_SCALE**2

# Per-epoch unique-token counts and completed epochs for Figure 2 pre-training.
CHECKPOINT_PRETRAINING = {
    "43M": {"unique_tokens_per_epoch": 5 * 5_120_000_000, "epochs": 20},
    "200M": {"unique_tokens_per_epoch": 5_120_000_000, "epochs": 50},
}

# Figure 2 checkpoints are shown on scaling plots but never passed to a fit.
EXCLUDED_SCALING_POINTS = pd.DataFrame(
    [
        {
            "model": "43M",
            "training_flops": 1.89131390976e20,
            "params_m": 42.890884,
            "tokens": CHECKPOINT_PRETRAINING["43M"]["unique_tokens_per_epoch"]
            * CHECKPOINT_PRETRAINING["43M"]["epochs"],
            "marker": "s",
        },
        {
            "model": "200M",
            "training_flops": 3.45362989056e20,
            "params_m": 199.938308,
            "tokens": CHECKPOINT_PRETRAINING["200M"]["unique_tokens_per_epoch"]
            * CHECKPOINT_PRETRAINING["200M"]["epochs"],
            "marker": "D",
        },
    ]
)

points = pd.read_csv(POINTS_CSV)
missing = REQUIRED_COLUMNS.difference(points.columns)
if missing:
    raise ValueError(f"Input CSV is missing columns: {sorted(missing)}")
if points[list(REQUIRED_COLUMNS)].isna().any().any():
    raise ValueError("Required numeric columns contain missing values")
if (points[list(REQUIRED_COLUMNS)] <= 0).any().any():
    raise ValueError("FLOPs, model sizes, iterations, and losses must be positive")
if (points.groupby("target_flops").size() < 3).any():
    raise ValueError("Each FLOP budget needs at least three points for a quadratic fit")
points["tokens"] = points["iter"] * TOKENS_PER_ITERATION


def quadratic_fit(x, y):
    """Fit log10(y) as a quadratic function of log10(x)."""
    log_x = np.log10(np.asarray(x, dtype=float))
    log_y = np.log10(np.asarray(y, dtype=float))
    coefficients = np.polyfit(log_x, log_y, deg=2)
    predicted = np.polyval(coefficients, log_x)
    residual_sum = np.square(log_y - predicted).sum()
    total_sum = np.square(log_y - log_y.mean()).sum()
    r_squared = 1.0 - residual_sum / total_sum if total_sum > 0 else 1.0
    return coefficients, float(r_squared)


def fit_budget(group, loss_column):
    """Fit one compute slice and return its best observed-range configuration."""
    group = group.sort_values("params_m")
    params = group["params_m"].to_numpy(dtype=float)
    tokens = group["tokens"].to_numpy(dtype=float)
    losses = group[loss_column].to_numpy(dtype=float)

    model_coefficients, model_r_squared = quadratic_fit(params, losses)
    token_coefficients, token_r_squared = quadratic_fit(tokens, losses)

    curvature, linear, _ = model_coefficients
    if curvature <= 0:
        raise ValueError(
            f"Non-convex {loss_column} fit at "
            f"{group['target_flops'].iloc[0]:.0e} FLOPs"
        )

    log_params = np.log10(params)
    unconstrained_log_vertex = -linear / (2.0 * curvature)
    optimum_log_params = float(
        np.clip(unconstrained_log_vertex, log_params.min(), log_params.max())
    )
    vertex_in_range = bool(
        log_params.min() <= unconstrained_log_vertex <= log_params.max()
    )

    # Each row in a fixed-compute slice pairs model size with training length.
    optimum_log_tokens = np.interp(
        optimum_log_params, log_params, np.log10(tokens)
    )
    optimum_log_loss = np.polyval(model_coefficients, optimum_log_params)
    optimum_tokens = 10.0**optimum_log_tokens

    return {
        "target_flops": float(group["target_flops"].iloc[0]),
        "n_points": len(group),
        "model_coefficients": model_coefficients,
        "token_coefficients": token_coefficients,
        "optimum_params_m": 10.0**optimum_log_params,
        "optimum_tokens": optimum_tokens,
        "optimum_iterations": optimum_tokens / TOKENS_PER_ITERATION,
        "fitted_min_loss": 10.0**optimum_log_loss,
        "unconstrained_params_m": 10.0**unconstrained_log_vertex,
        "vertex_in_observed_range": vertex_in_range,
        "model_fit_r2_log_loss": model_r_squared,
        "token_fit_r2_log_loss": token_r_squared,
    }


def format_flops(value):
    coefficient, exponent = f"{value:.0E}".split("E")
    return f"{coefficient}E{int(exponent)}"


print(f"Loaded {len(points)} points from {POINTS_CSV.relative_to(ROOT)}")
print(f"Patch tokens per image: {TOKENS_PER_IMAGE:,}")
print(f"Tokens per iteration: {TOKENS_PER_ITERATION:,}")
print(f"One epoch: {ONE_EPOCH_TOKENS:,} tokens")
points.groupby("target_flops").size().rename("points")

In [ ]:
def add_epoch_marker(axis, orientation="vertical"):
    """Mark one pass through the training data on a token axis."""
    marker_style = dict(color="0.72", linestyle="--", linewidth=1.4 * FIGURE_SCALE, zorder=0)
    if orientation == "vertical":
        axis.axvline(ONE_EPOCH_TOKENS, **marker_style)
        axis.annotate(
            "1 epoch",
            xy=(ONE_EPOCH_TOKENS, 0.98),
            xycoords=("data", "axes fraction"),
            xytext=(4, 0),
            textcoords="offset points",
            rotation=90,
            va="top",
            ha="left",
            color="0.45",
            fontsize=11 * FIGURE_SCALE,
        )
    elif orientation == "horizontal":
        axis.axhline(ONE_EPOCH_TOKENS, **marker_style)
        axis.annotate(
            "1 epoch",
            xy=(0.98, ONE_EPOCH_TOKENS),
            xycoords=("axes fraction", "data"),
            xytext=(0, 4),
            textcoords="offset points",
            va="bottom",
            ha="right",
            color="0.45",
            fontsize=11 * FIGURE_SCALE,
        )
    else:
        raise ValueError(f"Unknown marker orientation: {orientation}")


def plot_isoflop_metric(
    data,
    metric_name,
    loss_column,
    y_label,
    output_path,
    dpi=300,
):
    """Plot model-size and training-token views for one loss metric."""
    budgets = np.sort(data["target_flops"].unique())
    palette = sns.color_palette("mako_r", n_colors=len(budgets))
    summaries = []

    style = {
        "font.size": 13 * FIGURE_SCALE,
        "axes.labelsize": 18 * FIGURE_SCALE,
        "xtick.labelsize": 14 * FIGURE_SCALE,
        "ytick.labelsize": 14 * FIGURE_SCALE,
        "legend.fontsize": 13 * FIGURE_SCALE,
        "legend.title_fontsize": 14 * FIGURE_SCALE,
        "axes.linewidth": 1.6 * FIGURE_SCALE,
    }
    with plt.rc_context(style):
        figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE, sharey=True)

        for budget, color in zip(budgets, palette, strict=True):
            group = data[data["target_flops"] == budget].sort_values("params_m")
            fit = fit_budget(group, loss_column)
            summaries.append({"metric": metric_name, **fit})

            params = group["params_m"].to_numpy(dtype=float)
            tokens = group["tokens"].to_numpy(dtype=float)
            losses = group[loss_column].to_numpy(dtype=float)

            model_grid = np.geomspace(params.min(), params.max(), 300)
            token_grid = np.geomspace(tokens.min(), tokens.max(), 300)
            model_curve = 10.0 ** np.polyval(
                fit["model_coefficients"], np.log10(model_grid)
            )
            token_curve = 10.0 ** np.polyval(
                fit["token_coefficients"], np.log10(token_grid)
            )

            axes[0].plot(
                model_grid,
                model_curve,
                color=color,
                linewidth=2.2 * FIGURE_SCALE,
                label=format_flops(budget),
            )
            axes[0].scatter(params, losses, color=color, s=58 * MARKER_AREA_SCALE, zorder=3)
            axes[1].plot(token_grid, token_curve, color=color, linewidth=2.2 * FIGURE_SCALE)
            axes[1].scatter(tokens, losses, color=color, s=58 * MARKER_AREA_SCALE, zorder=3)


        axes[0].set_xscale("log")
        axes[1].set_xscale("log")
        axes[0].set_yscale("log")
        axes[0].yaxis.set_major_locator(ticker.LogLocator(base=10, subs=(1.0,)))
        axes[0].yaxis.set_major_formatter(
            ticker.FuncFormatter(lambda value, _: f"{value:g}")
        )
        axes[0].yaxis.set_minor_locator(
            ticker.LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
        )
        axes[0].yaxis.set_minor_formatter(ticker.NullFormatter())
        axes[0].set_xlabel("Model size (M)")
        axes[1].set_xlabel("Training tokens")
        axes[0].set_ylabel(y_label)
        axes[0].legend(title="FLOPs", loc="upper left", frameon=True)
        add_epoch_marker(axes[1], orientation="vertical")

        for axis in axes:
            axis.tick_params(which="major", direction="in", width=1.5 * FIGURE_SCALE, length=6 * FIGURE_SCALE)
            axis.tick_params(which="minor", direction="in", width=1.1 * FIGURE_SCALE, length=3 * FIGURE_SCALE)
            axis.margins(x=0.05)

        figure.tight_layout(w_pad=2.0)
        figure.savefig(output_path, dpi=dpi, transparent=True)
        plt.show()

    summary = pd.DataFrame(summaries)
    return summary.drop(columns=["model_coefficients", "token_coefficients"])

In [ ]:
summaries = []
for metric_name, (loss_column, y_label) in METRICS.items():
    output_path = PAPER_DIR / f"isoflop_{metric_name}.svg"
    summaries.append(
        plot_isoflop_metric(
            points,
            metric_name,
            loss_column,
            y_label,
            output_path,
            
        )
    )
    print(f"Wrote {output_path.relative_to(ROOT)}")

optima = pd.concat(summaries, ignore_index=True)
optima_path = PAPER_DIR / "isoflop_optima.csv"
optima.to_csv(optima_path, index=False)
print(f"Wrote {optima_path.relative_to(ROOT)}")

optima[
    [
        "metric",
        "target_flops",
        "optimum_params_m",
        "optimum_tokens",
        "fitted_min_loss",
        "vertex_in_observed_range",
        "model_fit_r2_log_loss",
    ]
].style.format(
    {
        "target_flops": "{:.0e}",
        "optimum_params_m": "{:.3f}",
        "optimum_tokens": "{:,.0f}",
        "fitted_min_loss": "{:.8f}",
        "model_fit_r2_log_loss": "{:.4f}",
    }
)

### Compute-optimal scaling within the measured budgets

Fit power laws to the interior IsoFLOP vertices,

$$
Y_{\mathrm{opt}} = A\left(\frac{C}{10^{18}}\right)^b,
$$

for optimal model size and optimal training tokens. Training- and validation-loss results are kept in separate figures because the training-loss optima are less reliable. The fitted lines are shown only across the FLOP range measured by the scaling experiments; no larger-compute projection is made.

Filled points are interior parabola vertices used for the power-law fits. Open points are optima clipped to the edge of the tested model-size range; they are displayed but excluded from the scaling fits. A hollow square and hollow diamond show the 43M and 200M Figure 2 checkpoints, respectively, using the fit color for their outlines; they are reference points only and are also excluded from fitting. The light-purple dashed lines show the Chinchilla equal-scaling exponent ($b=0.5$), normalized to each empirical fit at $10^{18}$ FLOPs so that the comparison isolates the slope rather than transferring a language-model-specific intercept, and span the full displayed FLOP range. The one-epoch token count is vertical when tokens are on the x-axis in the IsoFLOP profiles above, and horizontal when tokens are on the y-axis in the scaling figures below.

In [ ]:
COMPUTE_REFERENCE = 1e18
CHINCHILLA_EXPONENT = 0.5
CHINCHILLA_COLOR = "#b9a3d3"
SCALING_QUANTITIES = {
    "optimum_params_m": "Optimal model size (M)",
    "optimum_tokens": "Optimal training tokens",
}


def fit_power_law(compute, values, reference=COMPUTE_REFERENCE):
    """Fit values = value_at_reference * (compute / reference)**exponent."""
    log_compute = np.log10(np.asarray(compute, dtype=float) / reference)
    log_values = np.log10(np.asarray(values, dtype=float))
    exponent, log_value_at_reference = np.polyfit(log_compute, log_values, deg=1)
    predicted = exponent * log_compute + log_value_at_reference
    residual_sum = np.square(log_values - predicted).sum()
    total_sum = np.square(log_values - log_values.mean()).sum()
    r_squared = 1.0 - residual_sum / total_sum if total_sum > 0 else 1.0
    return {
        "value_at_1e18_flops": 10.0**log_value_at_reference,
        "exponent": float(exponent),
        "r2_log_log": float(r_squared),
    }


def plot_metric_scaling(
    metric_name,
    color,
    excluded_points=None,
    include_chinchilla=True,
    filename_suffix="",
    dpi=300,
):
    """Plot measured model-size and token optima for one loss metric."""
    metric_optima = optima[optima["metric"] == metric_name].sort_values(
        "target_flops"
    )
    interior = metric_optima[metric_optima["vertex_in_observed_range"]]
    boundary = metric_optima[~metric_optima["vertex_in_observed_range"]]
    metric_rows = []

    style = {
        "font.size": 13 * FIGURE_SCALE,
        "axes.labelsize": 18 * FIGURE_SCALE,
        "xtick.labelsize": 14 * FIGURE_SCALE,
        "ytick.labelsize": 14 * FIGURE_SCALE,
        "legend.fontsize": 11 * FIGURE_SCALE,
        "legend.title_fontsize": 12 * FIGURE_SCALE,
        "axes.linewidth": 1.6 * FIGURE_SCALE,
    }
    with plt.rc_context(style):
        figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE)

        for axis, (quantity, y_label) in zip(
            axes, SCALING_QUANTITIES.items(), strict=True
        ):
            fit = fit_power_law(interior["target_flops"], interior[quantity])
            metric_rows.append(
                {
                    "metric": metric_name,
                    "quantity": quantity,
                    "n_fit_points": len(interior),
                    **fit,
                }
            )

            compute_grid = np.geomspace(
                metric_optima["target_flops"].min(),
                metric_optima["target_flops"].max(),
                300,
            )
            fitted_values = fit["value_at_1e18_flops"] * (
                compute_grid / COMPUTE_REFERENCE
            ) ** fit["exponent"]

            axis.plot(
                compute_grid,
                fitted_values,
                color=color,
                linewidth=2.4 * FIGURE_SCALE,
                label=f"Power-law fit ($b={fit['exponent']:.2f}$)",
            )
            axis.scatter(
                interior["target_flops"],
                interior[quantity],
                color=color,
                s=70 * MARKER_AREA_SCALE,
                zorder=3,
                label="Interior optimum",
            )
            if not boundary.empty:
                axis.scatter(
                    boundary["target_flops"],
                    boundary[quantity],
                    facecolors="none",
                    edgecolors=[color],
                    linewidths=2.0 * FIGURE_SCALE,
                    s=75 * MARKER_AREA_SCALE,
                    zorder=3,
                    label="Boundary optimum (excluded)",
                )

            if excluded_points is not None:
                point_value_column = (
                    "params_m" if quantity == "optimum_params_m" else "tokens"
                )
                for point in excluded_points.itertuples(index=False):
                    axis.scatter(
                        point.training_flops,
                        getattr(point, point_value_column),
                        marker=point.marker,
                        s=105 * MARKER_AREA_SCALE,
                        facecolors="none",
                        edgecolors=[color],
                        linewidths=2.0 * FIGURE_SCALE,
                        zorder=5,
                        label=f"{point.model} checkpoint (excluded)",
                    )

            axis.set_xscale("log")
            axis.set_yscale("log")
            axis.set_xlabel("Training FLOPs")
            axis.set_ylabel(y_label)
            axis.tick_params(which="major", direction="in", width=1.5 * FIGURE_SCALE, length=6 * FIGURE_SCALE)
            axis.tick_params(which="minor", direction="in", width=1.1 * FIGURE_SCALE, length=3 * FIGURE_SCALE)
            axis.margins(x=0.05, y=0.12)
            if include_chinchilla:
                displayed_flop_range = axis.get_xlim()
                reference_compute_grid = np.geomspace(
                    *displayed_flop_range, 300
                )
                chinchilla_values = fit["value_at_1e18_flops"] * (
                    reference_compute_grid / COMPUTE_REFERENCE
                ) ** CHINCHILLA_EXPONENT
                axis.plot(
                    reference_compute_grid,
                    chinchilla_values,
                    color=CHINCHILLA_COLOR,
                    linestyle="--",
                    linewidth=2.2 * FIGURE_SCALE,
                    zorder=1,
                    label="Chinchilla reference ($b=0.5$)",
                )
                axis.set_xlim(displayed_flop_range)
            axis.legend(loc="best", frameon=True)

        add_epoch_marker(axes[1], orientation="horizontal")
        figure.tight_layout(w_pad=2.5)
        figure_path = (
            PAPER_DIR
            / f"isoflop_{metric_name}_optimal_scaling{filename_suffix}.svg"
        )
        figure.savefig(figure_path, dpi=dpi, transparent=True)
        plt.show()

    return pd.DataFrame(metric_rows), figure_path


scaling_tables = []
metric_colors = {
    "validation": sns.color_palette("mako_r", n_colors=3)[-1],
    "training": sns.color_palette("mako_r", n_colors=3)[0],
}
for metric_name in ("validation", "training"):
    metric_fits, figure_path = plot_metric_scaling(
        metric_name,
        metric_colors[metric_name],
        excluded_points=(
            EXCLUDED_SCALING_POINTS
            if metric_name == "validation"
            else None
        ),
    )
    scaling_tables.append(metric_fits)
    print(f"Wrote {figure_path.relative_to(ROOT)}")

# Clean main-text variant: retain fitted optima but omit external references.
_, clean_figure_path = plot_metric_scaling(
    "validation",
    metric_colors["validation"],
    excluded_points=None,
    include_chinchilla=False,
    filename_suffix="_no_references",
)
print(f"Wrote {clean_figure_path.relative_to(ROOT)}")

scaling_fits = pd.concat(scaling_tables, ignore_index=True)
scaling_fits_path = PAPER_DIR / "isoflop_scaling_fits.csv"
scaling_fits.to_csv(scaling_fits_path, index=False)
print(f"Wrote {scaling_fits_path.relative_to(ROOT)}")

scaling_fits.style.format(
    {
        "value_at_1e18_flops": "{:.4g}",
        "exponent": "{:.4f}",
        "r2_log_log": "{:.4f}",
    }
)

### Figure 2 model training compute

Compute the pre-training and experiment-specific fine-tuning FLOPs for the 43M and 200M models used in Figure 2. PtychoViTFlopsCalculator counts the analytical forward pass through the custom ViT and twin decoders using two FLOPs per multiply-add. Following the training estimate in utils/flops_utils.py, backward propagation is approximated as twice the forward pass, so one training example costs three analytical forward passes. This does not include the physics-forward loss or optimizer update.

For pre-training, the 43M model used five groups of 5.12 billion unique tokens per epoch for 20 epochs, while the 200M model used 5.12 billion unique tokens per epoch for 50 epochs. The cumulative token totals determine the pre-training pattern and FLOP counts below. Both Figure 2 fine-tunes used 101 epochs on 44,827 training patterns across 24 ranks; PyTorch's distributed sampler pads that split to 44,832 processed patterns per epoch. Both architectures are specified directly below, and the historical 200M standard decoders explicitly retain base_channels=64.

In [ ]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.flops_utils import PtychoViTFlopsCalculator


def make_model_config(embed_dim, depth, num_heads, base_channels):
    return {
        "encoder_type": "custom",
        "encoder": {
            "embed_dim": embed_dim,
            "depth": depth,
            "num_heads": num_heads,
        },
        "decoder": {
            "base_channels": base_channels,
            "latent_dim": None,
        },
    }


model_configs = {
    "43M": make_model_config(512, 12, 8, 64),
    "200M": make_model_config(896, 20, 14, 64),
}

# Reuse the cumulative-token histories plotted in the scaling figure.
pretraining_history = CHECKPOINT_PRETRAINING

# Runs 164 and 167 used the same fine-tuning workload. DistributedSampler
# adds five repeated examples so every one of the 24 ranks has equal work.
FINETUNE_EPOCHS = 101
FINETUNE_TRAIN_PATTERNS = 44_827
FINETUNE_WORLD_SIZE = 24
finetune_patterns_per_epoch = (
    (FINETUNE_TRAIN_PATTERNS + FINETUNE_WORLD_SIZE - 1)
    // FINETUNE_WORLD_SIZE
    * FINETUNE_WORLD_SIZE
)

FORWARD_TO_TRAINING_FACTOR = 3.0
rows = []
for model_name, model_config in model_configs.items():
    calculator = PtychoViTFlopsCalculator(model_config, batch_size=1)
    forward_flops = calculator.flops_analytical() * 1e12
    training_flops_per_pattern = FORWARD_TO_TRAINING_FACTOR * forward_flops
    history = pretraining_history[model_name]
    pretraining_tokens = (
        history["unique_tokens_per_epoch"] * history["epochs"]
    )
    if pretraining_tokens % TOKENS_PER_IMAGE:
        raise ValueError("Pre-training tokens must map to whole input patterns")
    pretraining_patterns = pretraining_tokens // TOKENS_PER_IMAGE
    finetuning_patterns = finetune_patterns_per_epoch * FINETUNE_EPOCHS
    encoder, decoder = model_config["encoder"], model_config["decoder"]
    rows.append(
        {
            "model": model_name,
            "params_m": calculator.param_count(),
            "embed_dim": encoder["embed_dim"],
            "depth": encoder["depth"],
            "num_heads": encoder["num_heads"],
            "base_channels": decoder["base_channels"],
            "forward_flops_per_pattern": forward_flops,
            "training_flops_per_pattern": training_flops_per_pattern,
            "pretraining_tokens": pretraining_tokens,
            "pretraining_patterns": pretraining_patterns,
            "pretraining_flops": training_flops_per_pattern
            * pretraining_patterns,
            "finetuning_patterns": finetuning_patterns,
            "finetuning_flops": training_flops_per_pattern
            * finetuning_patterns,
        }
    )

fig2_model_compute = pd.DataFrame(rows).set_index("model")
plotted_checkpoint_compute = EXCLUDED_SCALING_POINTS.set_index("model")
np.testing.assert_allclose(
    fig2_model_compute["pretraining_tokens"],
    plotted_checkpoint_compute["tokens"],
)
np.testing.assert_allclose(
    fig2_model_compute["pretraining_flops"],
    plotted_checkpoint_compute["training_flops"],
)

display_columns = [
    "params_m",
    "embed_dim",
    "depth",
    "num_heads",
    "base_channels",
    "forward_flops_per_pattern",
]

def format_compute_table(columns):
    formatters = {
        column: (
            "{:.3e}"
            if "flops" in column
            else "{:.3f}"
            if column == "params_m"
            else "{:,}"
        )
        for column in columns
    }
    return fig2_model_compute[columns].style.format(formatters)


print("Pre-training FLOPs")
pretraining_columns = display_columns + [
    "pretraining_tokens",
    "pretraining_patterns",
    "pretraining_flops",
]
display(format_compute_table(pretraining_columns))
print("Fine-tuning FLOPs")
finetuning_columns = display_columns + [
    "finetuning_patterns",
    "finetuning_flops",
]
display(format_compute_table(finetuning_columns))

fig2_model_compute